# From Tokens to Meaning: A Visual NLP Demo

Live seminar notebook: tokenization → vectors → embeddings → Word2Vec → vector space.

Toy coordinates are illustrative; trained Word2Vec vectors are learned from the tiny corpus.

## 0. Setup

If needed, run `%pip install gensim scikit-learn`.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=3, suppress=True)


## 1. Tokenization

This regex tokenizer is a teaching simplification. Production LLMs often use subword tokenizers.

In [ ]:
sentence = "The animal didn't cross the street because it was tired."
tokens = re.findall(r"\w+|[^\w\s]", sentence.lower())
print("Text:", sentence)
print("Tokens:", tokens)
print("Token count:", len(tokens))


In [ ]:
vocab = {token: idx for idx, token in enumerate(dict.fromkeys(tokens))}
token_ids = [vocab[t] for t in tokens]
print("Vocabulary:", vocab)
print("Token IDs:", token_ids)


## 2. One-hot vectors: IDs are not meaning

A one-hot vector marks a vocabulary position; it does not encode semantic similarity.

In [ ]:
def one_hot(token, vocab):
    vec = np.zeros(len(vocab))
    vec[vocab[token]] = 1
    return vec

for word in ["animal", "street", "tired"]:
    print(word, one_hot(word, vocab))


## 3. Vector space

Made-up 2D coordinates make geometry visible; these are not learned model values.

In [ ]:
toy_vectors = {
    "cat": np.array([.9,.8]), "dog": np.array([.85,.75]),
    "puppy": np.array([.8,.7]), "car": np.array([-.7,.6]),
    "truck": np.array([-.8,.55]), "apple": np.array([.1,-.8]),
    "banana": np.array([.2,-.75])
}
plt.figure(figsize=(8,5))
for word, v in toy_vectors.items():
    plt.scatter(v[0], v[1], s=65)
    plt.annotate(word, (v[0]+.025, v[1]+.025), fontsize=11)
plt.axhline(0, linewidth=.7); plt.axvline(0, linewidth=.7)
plt.title("Illustrative word vectors in 2D")
plt.xlabel("Dimension 1"); plt.ylabel("Dimension 2")
plt.grid(alpha=.25); plt.show()


## 4. Cosine similarity

Cosine similarity compares vector direction, commonly used to compare embeddings.

In [ ]:
def cosine(a,b):
    d=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/d) if d else 0.0

for a,b in [("cat","dog"),("cat","car"),("apple","banana")]:
    print(f"cos({a}, {b}) = {cosine(toy_vectors[a],toy_vectors[b]):.3f}")


## 5. Word2Vec: learn from context

Word2Vec learns vectors from nearby words. This tiny corpus demonstrates the workflow; results are unstable and not representative of a large model.

In [ ]:
# If needed: %pip install gensim
try:
    from gensim.models import Word2Vec
    texts = [
        "the cat sat on the mat", "the dog sat on the rug",
        "a puppy is a young dog", "the kitten is a young cat",
        "the car drove on the road", "the truck drove on the highway",
        "an apple is a fruit", "a banana is a fruit"
    ]
    corpus = [s.split() for s in texts] * 30
    w2v = Word2Vec(sentences=corpus, vector_size=20, window=3,
                   min_count=1, workers=1, sg=1, seed=7, epochs=100)
    print("Vector for cat:", w2v.wv["cat"])
    print("Similar to cat:", w2v.wv.most_similar("cat", topn=5))
except ImportError:
    print("gensim missing: run %pip install gensim and rerun this cell")


## 6. Plot Word2Vec vectors (PCA)

PCA projects learned 20D vectors into 2D; some information is lost.

In [ ]:
try:
    from sklearn.decomposition import PCA
    words = [w for w in ["cat","dog","puppy","kitten","car","truck","apple","banana","fruit"] if w in w2v.wv]
    matrix = np.array([w2v.wv[w] for w in words])
    coords = PCA(n_components=2).fit_transform(matrix)
    plt.figure(figsize=(8,5)); plt.scatter(coords[:,0],coords[:,1],s=70)
    for word,(x,y) in zip(words,coords):
        plt.annotate(word,(x+.02,y+.02),fontsize=11)
    plt.title("Tiny Word2Vec model: PCA projection")
    plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.grid(alpha=.25); plt.show()
except (NameError, ImportError) as e:
    print("Train Word2Vec first; install scikit-learn if needed.", e)


## 7. Embedding lookup: token → dense vector

An embedding table maps a token ID to a learned dense vector. Word2Vec lookup is a useful analogy.

In [ ]:
if "w2v" in globals():
    word = "cat"
    print("Token:", word)
    print("Dense embedding:", w2v.wv[word])
    print("Dimensions:", len(w2v.wv[word]))


## 8. Takeaway

**Text → tokens → token IDs → embedding lookup → vectors → model layers → contextual representations**

Word2Vec typically assigns a static vector per word. Transformers build contextual representations, so a word's representation depends on its sentence.